# Reproducible one-step traffic forecasting experiment

This notebook is the executable report for the comparison described in
`EXPERIMENT_PROTOCOL.md`.

## What is being compared

All accuracy values in this notebook use the same **traffic-speed-only** input,
the same chronological split, the same target timestamps, and the same
**one-step (five-minute)** horizon. The weather columns are not used in the
primary comparison. TimesFM is a pretrained zero-shot model; the LSTM and the
DSS-softmax traffic wrapper are supervised models trained on the training split. The notebook therefore reports common evaluation conditions
without claiming that the models have identical training histories.

The baseline is a traffic-specific **DSS-softmax** wrapper based on the
canonical diagonal state-space recurrence. The code and protocol document its
complex-mode initialization, length-normalized ZOH coefficients, and exact
trainable parameter count. The original custom gated recurrence remains
available under an explicit DSS-inspired alias for a separate ablation.

In [ ]:
from pathlib import Path
import os
import sys

# Resolve the repository even if Jupyter was opened from a different directory.
ROOT = Path.cwd().resolve()
while not (ROOT / "traffic_experiment.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "traffic_experiment.py").exists():
    raise FileNotFoundError("Could not locate traffic_experiment.py")

sys.path.insert(0, str(ROOT))

from traffic_experiment import ExperimentConfig, run_experiment

print("Repository:", ROOT)

## Run configuration

The default is the full protocol. For a quick execution check, temporarily set
`FAST_MODE = True`; fast-mode outputs are marked as smoke-test results and must
not be reported as final measurements. Set `RUN_TIMESFM = False` only
when the checkpoint is unavailable; the notebook will record that limitation
instead of inventing a TimesFM result.

In [ ]:
FAST_MODE = os.environ.get("EXPERIMENT_FAST", "0") == "1"
RUN_TIMESFM = os.environ.get("EXPERIMENT_RUN_TIMESFM", "1") == "1"

config = ExperimentConfig(
    output_dir=ROOT / "outputs",
    fast=FAST_MODE,
    run_timesfm=RUN_TIMESFM,
)
results = run_experiment(**config.to_kwargs())

results["metrics"]

## Measured results and reproducibility record

The run writes `metrics.csv`, `efficiency.csv`, `run_metadata.json`, and
unit-separated figures under `outputs/`. The tables include sample counts,
parameter counts, device, and timing. The report-facing `mae_rmse_speed.png`
and `mape_percent.png` files keep speed-unit and percentage errors visually
separate. Use the generated files as the source for reported tables; do not
copy numbers from an earlier notebook execution.

In [ ]:
import json
from IPython.display import Image, display

metrics = results["metrics"]
efficiency = results["efficiency"]
metadata = results["metadata"]

print("Experiment metadata")
print(json.dumps(metadata, indent=2, default=str))
print("\nAccuracy metrics")
print(metrics.to_string(index=False))
print("\nEfficiency metrics")
print(efficiency.to_string(index=False))

figure_names = ["mae_rmse_speed.png", "mape_percent.png", "metrics_by_unit.png"]
for figure_name in figure_names:
    figure_path = ROOT / "outputs" / "figures" / figure_name
    if figure_path.exists():
        display(Image(filename=str(figure_path)))

## Interpretation rules

- Report MAE and RMSE in traffic-speed units and MAPE as a percentage.
- Describe the primary task as one-step-ahead traffic-speed forecasting.
- Do not describe the three models as having identical training conditions:
  TimesFM is zero-shot, while the other two models are trained on the training
  split.
- Do not describe the DSS-softmax traffic wrapper as a reproduction of the
  original DSS benchmark code; state the recurrence and initialization explicitly.
- Use `efficiency.csv` for RQ2 timing and memory claims. Training time is not
  applicable to the zero-shot TimesFM model and should be marked as such.
- The first TimesFM execution downloads its checkpoint and can be slow on CPU.